# 01 — AlphaEarth Embedding PCA Exploration

Fits PCA on the 64-dimensional AlphaEarth Foundations embeddings across all 2017 ice pixels, and produces the scree plot used in the paper (Figure 1) to motivate the use of raw embeddings (rather than a reduced PCA basis) as MLP/CNN input features.

All logic lives in `glacier_melt.sampling` and `glacier_melt.visualise`; this notebook only loads data, calls those functions, and displays results.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt

from glacier_melt.sampling import load_parquet_regions, fit_pca, sanity_check
from glacier_melt.visualise import plot_pca_scree

DATA_DIR = Path("../data/Training_Data_2017")  # adjust if parquets are stored elsewhere
OUTPUT_DIR = Path("../results")
OUTPUT_DIR.mkdir(exist_ok=True)

## Load 2017 training data

Loads all four regions (R1a, R1b, R2, R3) and runs basic sanity checks (null counts, AE band variance, ice/melt class separation).

In [ ]:
df = load_parquet_regions(DATA_DIR)
sanity_check(df)

## Fit PCA on AlphaEarth embeddings

Fits a `StandardScaler` followed by PCA across all 64 embedding dimensions, using every 2017 ice pixel (Peru-wide). The fitted PCA and scaler are saved for use elsewhere in the pipeline (e.g. for the RF + AE PCA baseline model).

In [ ]:
pca, scaler = fit_pca(
    df,
    n_components=64,
    save_path=OUTPUT_DIR / "ae_pca",
)

## Scree plot (Figure 1)

Shows that no small number of principal components captures the majority of variance in the AlphaEarth embedding space — motivating the decision to use all 64 raw dimensions as MLP/CNN input rather than a reduced PCA basis.

In [ ]:
fig = plot_pca_scree(pca, n_components_highlight=10)
fig.savefig(OUTPUT_DIR / "pca_scree.png", dpi=200, bbox_inches="tight")
plt.show()